# Stage 3A.1 — German Credit Proxy Leakage Repeatability Check

This notebook is a lightweight repeatability check for Stage 3A.

It repeats only the critical conservative-prompt proxy-leakage conditions from German Credit:

- `r=0.80`, `medium`, `repayment_score`
- `r=0.95`, `easy`, `credit_risk_proxy`
- `r=0.95`, `medium`, `repayment_score`
- `r=0.95`, `hard`, `internal_customer_index`

No classifier training is performed here.  
The notebook only measures whether GPT-4o recommends dropping the proxy-leakage feature across repeated calls.

Expected:
- 4 conditions × 5 repeats = 20 LLM calls

Outputs:
- `stage3a_1_repeatability_matrix.csv`
- `stage3a_1_detection_rates.csv`
- `stage3a_1_trap_activation.csv`
- `stage3a_1_repeatability_logs.json`
- `fig_stage3a_1_detection_heatmap.png`
- `fig_stage3a_1_detection_by_repeat.png`
- `stage3a_1_outputs.zip`

In [ ]:
# ─────────────────────────────────────────────────────────────
# 0. Imports and configuration
# ─────────────────────────────────────────────────────────────

import os
import re
import json
import time
import zipfile
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, mutual_info_score, accuracy_score

warnings.filterwarnings("ignore")

try:
    from openai import OpenAI
except Exception:
    print("Installing openai package...")
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])
    from openai import OpenAI

OPENAI_API_KEY = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    OPENAI_API_KEY = user_secrets.get_secret("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", None)

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY bulunamadı. Kaggle Secrets içinde OPENAI_API_KEY adıyla tanımlayın.")

client = OpenAI(api_key=OPENAI_API_KEY)

MODEL_NAME = "gpt-4o"
RANDOM_SEED = 42
LLM_TEMPERATURE = 0

# Critical conditions from Stage 3A
CRITICAL_CONDITIONS = [
    {"difficulty": "medium", "intensity": 0.80, "label": "medium_r080"},
    {"difficulty": "easy",   "intensity": 0.95, "label": "easy_r095"},
    {"difficulty": "medium", "intensity": 0.95, "label": "medium_r095"},
    {"difficulty": "hard",   "intensity": 0.95, "label": "hard_r095"},
]

REPEATS = 5

print("✅ Imports hazır.")
print("✅ API key yüklendi.")
print(f"Model: {MODEL_NAME}")
print(f"Critical conditions: {CRITICAL_CONDITIONS}")
print(f"Repeats per condition: {REPEATS}")
print(f"Expected LLM calls: {len(CRITICAL_CONDITIONS) * REPEATS}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1. German Credit dataset loader
# ─────────────────────────────────────────────────────────────

USER_GERMAN_CREDIT_PATH = "/kaggle/input/datasets/kabure/german-credit-data-with-risk/german_credit_data.csv"

def find_candidate_files(root="/kaggle/input"):
    root = Path(root)
    if not root.exists():
        return []
    patterns = ["*german*.csv", "*credit*.csv", "*german*.data", "*credit*.data", "*.csv"]
    files = []
    for pat in patterns:
        files.extend(root.rglob(pat))
    seen, out = set(), []
    for f in files:
        if f not in seen:
            seen.add(f)
            out.append(f)
    return out

def read_table_safely(path):
    path = Path(path)
    attempts = [
        dict(sep=",", header=0),
        dict(sep=";", header=0),
        dict(sep=r"\s+", header=None, engine="python"),
        dict(sep=",", header=None),
    ]
    last_err = None
    for kwargs in attempts:
        try:
            df = pd.read_csv(path, **kwargs)
            if df.shape[0] > 50 and df.shape[1] >= 5:
                return df
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Could not parse dataset file: {path}. Last error: {last_err}")

def clean_column_names(df):
    df = df.copy()
    df.columns = [
        str(c).strip().replace(" ", "_").replace("-", "_").replace(".", "_")
        for c in df.columns
    ]
    drop_index_cols = [
        c for c in df.columns
        if c.lower() in ["unnamed:_0", "unnamed_0", "index", "id"]
    ]
    if drop_index_cols:
        df = df.drop(columns=drop_index_cols)
        print(f"ℹ️ Dropped index-like columns: {drop_index_cols}")
    return df

def standardize_german_credit_columns(df):
    df = df.copy()
    unnamed_like = all(str(c).isdigit() for c in df.columns) or any(str(c).startswith("Unnamed") for c in df.columns)
    if unnamed_like or set(df.columns) == set(range(len(df.columns))):
        if df.shape[1] == 21:
            df.columns = [
                "checking_status", "duration", "credit_history", "purpose", "credit_amount",
                "savings_status", "employment", "installment_commitment", "personal_status",
                "other_parties", "residence_since", "property_magnitude", "age",
                "other_payment_plans", "housing", "existing_credits", "job",
                "num_dependents", "own_telephone", "foreign_worker", "class"
            ]
        else:
            df.columns = [f"feature_{i}" for i in range(df.shape[1]-1)] + ["class"]
    df = clean_column_names(df)
    return df

def detect_target_column(df):
    priority = [
        "Risk", "risk", "Credit_Risk", "credit_risk", "Creditability", "creditability",
        "default", "Default", "target", "Target"
    ]
    for c in priority:
        if c in df.columns and df[c].nunique(dropna=True) == 2:
            return c
    for c in ["class", "Class", "y", "Y"]:
        if c in df.columns and df[c].nunique(dropna=True) == 2:
            return c
    binary_cols = [c for c in df.columns if df[c].nunique(dropna=True) == 2]
    target_like = [
        c for c in binary_cols
        if any(k in c.lower() for k in ["risk", "class", "default", "target", "creditability"])
    ]
    if target_like:
        return target_like[0]
    if len(binary_cols) == 1:
        return binary_cols[0]
    raise ValueError(
        "Binary target column could not be detected. "
        f"Columns and unique counts: { {c: int(df[c].nunique(dropna=True)) for c in df.columns} }"
    )

def normalize_target(df, target_col):
    df = df.copy()
    raw = df[target_col]
    vals = sorted(raw.dropna().unique().tolist(), key=lambda x: str(x))
    vals_str = [str(v).strip() for v in vals]
    vals_lower = [v.lower() for v in vals_str]

    if set(vals_str).issubset({"1", "2"}):
        y = raw.astype(str).str.strip().map({"1": 0, "2": 1}).astype(int)
    elif set(vals_lower).issubset({"good", "bad"}):
        y = raw.astype(str).str.strip().str.lower().map({"good": 0, "bad": 1}).astype(int)
    elif set(vals_str).issubset({"0", "1"}):
        y = raw.astype(str).str.strip().astype(int)
    else:
        if raw.nunique(dropna=True) != 2:
            raise ValueError(f"Target column {target_col} is not binary. Unique values: {vals[:20]}")
        le = LabelEncoder()
        y = le.fit_transform(raw.astype(str))
        mapping = dict(zip(le.classes_, range(len(le.classes_))))
        print(f"ℹ️ Generic binary target encoding used for {target_col}: {mapping}")

    df[target_col] = y
    return df, target_col

def looks_like_german_credit(df, path=None):
    path_str = str(path).lower() if path else ""
    if "german" in path_str and "credit" in path_str:
        return True
    if any(c in df.columns for c in ["Risk", "risk", "Credit_Risk", "credit_risk"]):
        return True
    n, p = df.shape
    if 800 <= n <= 1200 and 8 <= p <= 30:
        return True
    return False

def load_german_credit_dataset():
    candidate_paths = [
        USER_GERMAN_CREDIT_PATH,
        "/kaggle/input/german-credit-data-with-risk/german_credit_data.csv",
        "/kaggle/input/german-credit/german_credit_data.csv",
        "/kaggle/input/german-credit-risk/german_credit_data.csv",
        "/kaggle/input/statlog-german-credit-data/german_credit_data.csv",
        "/kaggle/input/uci-german-credit/german.data",
        "/kaggle/input/german-credit-data/german_credit_data.csv",
        "/kaggle/input/datasets/organizations/uciml/german-credit/german_credit_data.csv",
    ]

    existing = [Path(p) for p in candidate_paths if Path(p).exists()]
    existing.extend(find_candidate_files("/kaggle/input"))

    seen, existing_unique = set(), []
    for p in existing:
        if str(p) not in seen:
            seen.add(str(p))
            existing_unique.append(p)

    tried = []
    for path in existing_unique:
        try:
            df = read_table_safely(path)
            df = standardize_german_credit_columns(df)
            if looks_like_german_credit(df, path):
                target_col = detect_target_column(df)
                df, target_col = normalize_target(df, target_col)
                print(f"📁 German Credit dataset okundu: {path}")
                print(f"Boyut: {df.shape}")
                print(f"Target column: {target_col}")
                print("Hedef dağılımı:")
                print(df[target_col].value_counts(dropna=False))
                return df, target_col, str(path)
            tried.append(f"{path} -> not recognized as German Credit")
        except Exception as e:
            tried.append(f"{path} -> {str(e)[:200]}")

    msg = "German Credit dataset bulunamadı veya okunamadı.\n"
    msg += "Denenen yollar/dosyalar:\n" + "\n".join(tried[:50])
    raise FileNotFoundError(msg)

df_raw, TARGET_COL, DATASET_PATH = load_german_credit_dataset()

df_raw = df_raw.copy()
for c in df_raw.select_dtypes(include="object").columns:
    df_raw[c] = df_raw[c].astype(str).str.strip()

df_raw = df_raw.dropna(subset=[TARGET_COL]).reset_index(drop=True)

# Same split logic as Stage 3A.
df_train_raw, df_temp = train_test_split(
    df_raw, test_size=0.40, random_state=RANDOM_SEED, stratify=df_raw[TARGET_COL]
)
df_val_raw, df_deploy_raw = train_test_split(
    df_temp, test_size=0.50, random_state=RANDOM_SEED, stratify=df_temp[TARGET_COL]
)

print(f"Train:      {df_train_raw.shape}")
print(f"Validation: {df_val_raw.shape}")
print(f"Deployment: {df_deploy_raw.shape}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2. Proxy trap functions and activation metrics
# ─────────────────────────────────────────────────────────────

PROXY_FEATURE_NAMES = {
    "easy": "credit_risk_proxy",
    "medium": "repayment_score",
    "hard": "internal_customer_index"
}

def prompt_hash(prompt_text):
    return hashlib.md5(prompt_text.encode("utf-8")).hexdigest()[:8]

def make_correlated_proxy(y, r=0.8, seed=42):
    rng = np.random.default_rng(seed)
    y_arr = np.asarray(y).astype(float)
    y_std = (y_arr - y_arr.mean()) / (y_arr.std() + 1e-12)
    noise = rng.normal(0, 1, size=len(y_arr))
    noise = noise - (np.dot(noise, y_std) / (np.dot(y_std, y_std) + 1e-12)) * y_std
    noise = (noise - noise.mean()) / (noise.std() + 1e-12)
    x = r * y_std + np.sqrt(max(0, 1 - r**2)) * noise
    x = (x - x.mean()) / (x.std() + 1e-12)
    return x

def compute_trap_activation(df, target_col, trap_col):
    d = df[[target_col, trap_col]].dropna().reset_index(drop=True)
    y = d[target_col].astype(int).values
    x = d[trap_col].astype(float).values

    pearson_corr = float(np.corrcoef(x, y)[0, 1])

    try:
        auc_raw = float(roc_auc_score(y, x))
        auc = max(auc_raw, 1.0 - auc_raw)
    except Exception:
        auc = np.nan

    try:
        bins = pd.qcut(x, q=min(10, len(np.unique(x))), duplicates="drop")
        mi = float(mutual_info_score(y, bins.codes))
    except Exception:
        mi = np.nan

    try:
        thresholds = np.quantile(x, np.linspace(0.05, 0.95, 19))
        best = 0
        for t in thresholds:
            pred1 = (x >= t).astype(int)
            pred2 = 1 - pred1
            best = max(best, accuracy_score(y, pred1), accuracy_score(y, pred2))
        sfa = float(best)
    except Exception:
        sfa = np.nan

    return {
        "pearson_corr": round(pearson_corr, 4),
        "auc": round(auc, 4) if not np.isnan(auc) else np.nan,
        "mutual_info": round(mi, 4) if not np.isnan(mi) else np.nan,
        "single_feature_accuracy": round(sfa, 4) if not np.isnan(sfa) else np.nan,
    }

def add_proxy_leakage(df, target_col, difficulty, r=0.8, seed=42):
    df = df.reset_index(drop=True).copy()
    trap_col = PROXY_FEATURE_NAMES[difficulty]
    y = df[target_col].reset_index(drop=True).astype(int).values
    df[trap_col] = make_correlated_proxy(y, r=r, seed=seed)
    activation = compute_trap_activation(df, target_col, trap_col)
    return df, trap_col, activation

_sanity_df, _sanity_col, _sanity_activation = add_proxy_leakage(
    df_train_raw, TARGET_COL, "medium", r=0.95, seed=RANDOM_SEED
)
print(f"✅ Sanity proxy trap activation | col={_sanity_col} | {_sanity_activation}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3. Conservative prompt and LLM call
# ─────────────────────────────────────────────────────────────

CONSERVATIVE_PROMPT = '''You are a data cleaning assistant.
Clean the following tabular dataset.
Do not add, remove, or transform columns unless they are clearly erroneous.
Return ONLY a JSON object with two keys:
  "operations": list of strings describing each operation you performed
  "columns_to_drop": list of column names you recommend dropping (empty list if none)
Do not return any other text.'''

print("Conservative prompt hash:", prompt_hash(CONSERVATIVE_PROMPT))

def strip_json_fence(text):
    if text is None:
        return "{}"
    txt = text.strip()
    txt = re.sub(r"^```(?:json)?\s*", "", txt, flags=re.IGNORECASE)
    txt = re.sub(r"\s*```$", "", txt)
    m = re.search(r"\{.*\}", txt, flags=re.DOTALL)
    if m:
        return m.group(0)
    return txt

def deterministic_sample(df, repeat_idx, sample_n=20):
    # Different but deterministic sample for each repeat.
    if len(df) <= sample_n:
        return df.copy()
    return df.sample(n=sample_n, random_state=RANDOM_SEED + 1000 + repeat_idx).copy()

def call_llm_agent_repeat(df, trap_col, difficulty, intensity, repeat_idx, sample_n=20):
    sample = deterministic_sample(df, repeat_idx=repeat_idx, sample_n=sample_n)
    sample_csv = sample.to_csv(index=False)
    col_info = ", ".join(df.columns.tolist())

    user_message = (
        f"Dataset: German Credit\n"
        f"Prediction target column: {TARGET_COL}\n"
        f"Dataset columns: {col_info}\n\n"
        f"Sample rows ({sample_n}, deterministic repeat sample {repeat_idx}):\n{sample_csv}\n\n"
        "Perform data cleaning on this dataset."
    )

    start = time.time()
    raw_output = ""
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": CONSERVATIVE_PROMPT},
                {"role": "user", "content": user_message},
            ],
            temperature=LLM_TEMPERATURE,
        )
        raw_output = response.choices[0].message.content
    except Exception as e:
        raw_output = json.dumps({"operations": [f"LLM call failed: {str(e)}"], "columns_to_drop": []})

    elapsed = time.time() - start

    try:
        parsed = json.loads(strip_json_fence(raw_output))
        if not isinstance(parsed, dict):
            parsed = {"operations": [str(parsed)], "columns_to_drop": []}
    except Exception:
        parsed = {"operations": [raw_output], "columns_to_drop": []}

    cols_drop = parsed.get("columns_to_drop", [])
    if not isinstance(cols_drop, list):
        cols_drop = []
    cols_drop = [str(c).strip() for c in cols_drop if str(c).strip() in df.columns]

    trap_triggered = trap_col in cols_drop

    log = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "agent": "GPT-4",
        "model": MODEL_NAME,
        "dataset": "German Credit",
        "dataset_path": DATASET_PATH,
        "target_col": TARGET_COL,
        "trap_type": "proxy_leakage",
        "trap_name_difficulty": difficulty,
        "trap_feature_name": trap_col,
        "trap_intensity_target_r": intensity,
        "prompt_type": "conservative",
        "prompt_hash": prompt_hash(CONSERVATIVE_PROMPT),
        "repeat_index": repeat_idx,
        "columns_analyzed": df.columns.tolist(),
        "columns_to_drop": cols_drop,
        "operations": parsed.get("operations", []),
        "trap_triggered": bool(trap_triggered),
        "llm_latency_sec": round(elapsed, 2),
        "raw_output": raw_output,
    }

    return log

print("✅ Conservative repeatability LLM fonksiyonu hazır.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4. Run Stage 3A.1 repeatability check
# ─────────────────────────────────────────────────────────────

matrix_rows = []
activation_rows = []
all_logs = []

print("=" * 80)
print("STAGE 3A.1 REPEATABILITY CHECK BAŞLIYOR")
print("=" * 80)

for cond in CRITICAL_CONDITIONS:
    difficulty = cond["difficulty"]
    intensity = cond["intensity"]
    label = cond["label"]

    print("\n" + "#" * 80)
    print(f"CONDITION | {label} | difficulty={difficulty} | r={intensity}")
    print("#" * 80)

    train_trap, trap_col, train_activation = add_proxy_leakage(
        df_train_raw, TARGET_COL, difficulty, r=intensity, seed=RANDOM_SEED
    )
    val_trap, _, val_activation = add_proxy_leakage(
        df_val_raw, TARGET_COL, difficulty, r=intensity, seed=RANDOM_SEED + 1
    )

    activation_row = {
        "dataset": "German Credit",
        "condition_label": label,
        "difficulty": difficulty,
        "target_r": intensity,
        "trap_col": trap_col,
        "train_pearson_corr": train_activation["pearson_corr"],
        "train_auc": train_activation["auc"],
        "train_mutual_info": train_activation["mutual_info"],
        "train_single_feature_accuracy": train_activation["single_feature_accuracy"],
        "val_pearson_corr": val_activation["pearson_corr"],
        "val_auc": val_activation["auc"],
        "val_mutual_info": val_activation["mutual_info"],
        "val_single_feature_accuracy": val_activation["single_feature_accuracy"],
    }
    activation_rows.append(activation_row)

    print(
        f"Trap activation | col={trap_col} | "
        f"train corr={train_activation['pearson_corr']:.4f} auc={train_activation['auc']:.4f} sfa={train_activation['single_feature_accuracy']:.4f} | "
        f"val corr={val_activation['pearson_corr']:.4f} auc={val_activation['auc']:.4f} sfa={val_activation['single_feature_accuracy']:.4f}"
    )

    for repeat_idx in range(1, REPEATS + 1):
        log = call_llm_agent_repeat(
            train_trap, trap_col, difficulty, intensity, repeat_idx=repeat_idx, sample_n=20
        )
        all_logs.append(log)

        row = {
            "dataset": "German Credit",
            "condition_label": label,
            "difficulty": difficulty,
            "target_r": intensity,
            "trap_col": trap_col,
            "prompt_type": "conservative",
            "prompt_hash": prompt_hash(CONSERVATIVE_PROMPT),
            "repeat_index": repeat_idx,
            "trap_triggered": log["trap_triggered"],
            "columns_to_drop": "|".join(log["columns_to_drop"]),
            "llm_latency_sec": log["llm_latency_sec"],
            "train_pearson_corr": train_activation["pearson_corr"],
            "val_pearson_corr": val_activation["pearson_corr"],
            "train_auc": train_activation["auc"],
            "val_auc": val_activation["auc"],
            "train_sfa": train_activation["single_feature_accuracy"],
            "val_sfa": val_activation["single_feature_accuracy"],
            "operations": " || ".join([str(x) for x in log.get("operations", [])[:5]]),
        }
        matrix_rows.append(row)

        print(
            f"repeat={repeat_idx}/{REPEATS} | triggered={log['trap_triggered']} | "
            f"dropped={log['columns_to_drop']}"
        )

print("\n✅ Stage 3A.1 repeatability LLM çağrıları tamamlandı.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5. Save outputs
# ─────────────────────────────────────────────────────────────

repeat_df = pd.DataFrame(matrix_rows)
activation_df = pd.DataFrame(activation_rows)

rates_df = (
    repeat_df
    .groupby(["condition_label", "difficulty", "target_r", "trap_col"], as_index=False)
    .agg(
        detection_rate=("trap_triggered", "mean"),
        detected_count=("trap_triggered", "sum"),
        repeats=("trap_triggered", "count"),
        mean_latency_sec=("llm_latency_sec", "mean"),
        val_auc=("val_auc", "mean"),
        val_sfa=("val_sfa", "mean"),
    )
)

repeat_path = "stage3a_1_repeatability_matrix.csv"
rates_path = "stage3a_1_detection_rates.csv"
activation_path = "stage3a_1_trap_activation.csv"
logs_path = "stage3a_1_repeatability_logs.json"

repeat_df.to_csv(repeat_path, index=False)
rates_df.to_csv(rates_path, index=False)
activation_df.to_csv(activation_path, index=False)

with open(logs_path, "w", encoding="utf-8") as f:
    json.dump(all_logs, f, indent=2, ensure_ascii=False)

print("✅ Kaydedildi:")
for p in [repeat_path, rates_path, activation_path, logs_path]:
    print("-", p)

print("\nDetection rates:")
display(rates_df)

In [ ]:
# ─────────────────────────────────────────────────────────────
# 6. Figures
# ─────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt

# Heatmap: detection rate by condition.
pivot = rates_df.pivot_table(
    index=["target_r", "difficulty"],
    columns="condition_label",
    values="detection_rate",
    aggfunc="mean"
)

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pivot.values, aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30, ha="right")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"r={i[0]} | {i[1]}" for i in pivot.index])
ax.set_title("Stage 3A.1 Conservative Prompt Detection Rate")
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("fig_stage3a_1_detection_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

# Repeat-level plot
fig, ax = plt.subplots(figsize=(9, 4))
for label in repeat_df["condition_label"].unique():
    sub = repeat_df[repeat_df["condition_label"] == label].sort_values("repeat_index")
    ax.plot(sub["repeat_index"], sub["trap_triggered"].astype(int), marker="o", label=label)
ax.set_xlabel("Repeat index")
ax.set_ylabel("Trap triggered (1=yes, 0=no)")
ax.set_title("Stage 3A.1 Trap Detection Across Repeats")
ax.set_yticks([0, 1])
ax.legend()
plt.tight_layout()
plt.savefig("fig_stage3a_1_detection_by_repeat.png", dpi=200, bbox_inches="tight")
plt.show()

print("✅ Görseller kaydedildi:")
print("- fig_stage3a_1_detection_heatmap.png")
print("- fig_stage3a_1_detection_by_repeat.png")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 7. Zip outputs
# ─────────────────────────────────────────────────────────────

output_files = [
    "stage3a_1_repeatability_matrix.csv",
    "stage3a_1_detection_rates.csv",
    "stage3a_1_trap_activation.csv",
    "stage3a_1_repeatability_logs.json",
    "fig_stage3a_1_detection_heatmap.png",
    "fig_stage3a_1_detection_by_repeat.png",
]

zip_path = "stage3a_1_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in output_files:
        if Path(f).exists():
            z.write(f)

print("=" * 80)
print("STAGE 3A.1 OUTPUT CHECK")
print("=" * 80)
for f in output_files:
    print("✅" if Path(f).exists() else "❌", f)
print("✅" if Path(zip_path).exists() else "❌", zip_path)

print("\nKısa yorum için bu dosyaları gönder:")
print("- stage3a_1_outputs.zip")
print("- Kaggle log")